Setting up a safe and standarized library to keep consistency

In [1]:
   import sys
   print(sys.executable)

C:\Users\zyany\GitHub\CENACE-CHALLENGE\arkham-env\Scripts\python.exe


All looking good

Setting up API to get updated information from site

In [2]:
import json, time
from datetime import date, timedelta
from pathlib import Path
import requests
import yaml
from tenacity import retry, stop_after_attempt, wait_exponential
from tqdm import tqdm

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

api   = cfg["api"]
nodes = cfg["nodes"]
start = date.fromisoformat(cfg["period"]["start"])
end   = date.fromisoformat(cfg["period"]["end"])

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(len(nodes), "nodes |", start, "→", end)

9 nodes | 2024-01-01 → 2025-06-30


Setting up the function to iterate through the api url requests 

In [3]:
def build_url(nodes, d_start, d_end):
    nodos = ",".join(nodes)          # comma-separated, no spaces (manual's rule)
    return (f"{api['base_url']}/{api['sistema']}/{api['proceso']}/{nodos}/"
            f"{d_start:%Y/%m/%d}/{d_end:%Y/%m/%d}/{api['formato']}")

print(build_url(nodes, start, start))

https://ws01.cenace.gob.mx:8082/SWPML/SIM/SIN/MDA/06AEO-115,06AHM-400,06FRO-115,06GRA-115,06MON-115,06MON-230,06RGS-115,06TKS-115,06XOC-115/2024/01/01/2024/01/01/JSON


Making a test call to go through the data and make sure functions are properly working

In [4]:
r = requests.get(build_url(nodes, start, start), timeout=120)
print("Status:", r.status_code)
test = r.json()
print(test.keys() if isinstance(test, dict) else type(test))

Status: 200
dict_keys(['nombre', 'proceso', 'sistema', 'area', 'Resultados', 'status'])


In [5]:
type(test)

dict

information structure is a dictionay so now will call the keys to see what's inside the dictionary

In [6]:
test.keys()

dict_keys(['nombre', 'proceso', 'sistema', 'area', 'Resultados', 'status'])

The data we are actually interested in would be under the "Resultados" key, so will explore that one

In [7]:
resultados = test["Resultados"]
type(resultados), len(resultados)

(list, 9)

It is a list of 9 which match the 9 nodes, we will take a look at the 1st row

In [8]:
resultados[0].keys()

dict_keys(['clv_nodo', 'Valores'])

It's giving another dictionary, so, I will look at the structure to get a better glance of indentation and nesting

In [9]:
print(json.dumps(test, indent=2, ensure_ascii=False)[:2000])

{
  "nombre": "PML",
  "proceso": "MDA",
  "sistema": "SIN",
  "area": "PÚBLICA",
  "Resultados": [
    {
      "clv_nodo": "06AEO-115",
      "Valores": [
        {
          "fecha": "2024-01-01",
          "hora": "1",
          "pml": "279.92",
          "pml_ene": "370.71",
          "pml_per": "-43.69",
          "pml_cng": "-47.1"
        },
        {
          "fecha": "2024-01-01",
          "hora": "2",
          "pml": "279.21",
          "pml_ene": "311.75",
          "pml_per": "-32.54",
          "pml_cng": "0"
        },
        {
          "fecha": "2024-01-01",
          "hora": "3",
          "pml": "274.23",
          "pml_ene": "305.52",
          "pml_per": "-31.28",
          "pml_cng": "0"
        },
        {
          "fecha": "2024-01-01",
          "hora": "4",
          "pml": "264.01",
          "pml_ene": "294.23",
          "pml_per": "-30.22",
          "pml_cng": "0"
        },
        {
          "fecha": "2024-01-01",
          "hora": "5",
          

API test worked fine :) important to note that every value is text which will be an issue if left like this when working with the data
Will flatten the data so it translates well and can adjust to the table dataset properly where the type of data will be fixed as well

In [10]:
def parse_response(data):
    df = pd.json_normalize(data["Resultados"], record_path="Valores", meta="clv_nodo")

    price_cols = ["pml", "pml_ene", "pml_per", "pml_cng"]
    df[price_cols] = df[price_cols].apply(pd.to_numeric)
    df["hora"]  = pd.to_numeric(df["hora"])
    df["fecha"] = pd.to_datetime(df["fecha"])
    df["timestamp"] = df["fecha"] + pd.to_timedelta(df["hora"] - 1, unit="h")

    return df[["clv_nodo", "timestamp", "fecha", "hora"] + price_cols]

Now it's time to check the test's shape and how data is displayed

In [11]:
import pandas as pd

In [12]:
df_test = parse_response(test)

print(df_test.shape)                         # expect (216, 8)
print(df_test["clv_nodo"].value_counts())    # expect 24 per node
print(df_test.dtypes)                        # prices float64, hora int64, dates datetime64

diff = (df_test["pml_ene"] + df_test["pml_per"] + df_test["pml_cng"] - df_test["pml"]).abs()
print("Max component difference:", diff.max())   # expect ~0.01 or less

df_test.head()

(216, 8)
clv_nodo
06AEO-115    24
06AHM-400    24
06FRO-115    24
06GRA-115    24
06MON-115    24
06MON-230    24
06RGS-115    24
06TKS-115    24
06XOC-115    24
Name: count, dtype: int64
clv_nodo                str
timestamp    datetime64[us]
fecha        datetime64[us]
hora                  int64
pml                 float64
pml_ene             float64
pml_per             float64
pml_cng             float64
dtype: object
Max component difference: 0.010000000000047748


,clv_nodo,timestamp,fecha,hora,pml,pml_ene,pml_per,pml_cng
0,06AEO-115,2024-01-01 00:00:00,2024-01-01,1,279.92,370.71,-43.69,-47.1
1,06AEO-115,2024-01-01 01:00:00,2024-01-01,2,279.21,311.75,-32.54,0.0
2,06AEO-115,2024-01-01 02:00:00,2024-01-01,3,274.23,305.52,-31.28,0.0
3,06AEO-115,2024-01-01 03:00:00,2024-01-01,4,264.01,294.23,-30.22,0.0
4,06AEO-115,2024-01-01 04:00:00,2024-01-01,5,243.44,267.66,-24.22,0.0


Shape is 216,8 which is what was expected since we asked on the test to retrieve the data of 1 day which is 24hrs for each node, 9, and we asked for 8 columns
Each node has 24 value counts that align to value per hour of the day
Type of value for each category is alligned now
Max component difference was added since the prices might be calculated with numbers with many decimals and then publish them rounded to 2, we want to confirm consistency

Time to split the calls in chunks since CENACE's database only allows to retreive 7 days information and the given dates for this case
sum 547 days in total, so api call will have to iterate 79 times
Gotta make sure that the first chunk starts 2024-01-01, and last ends 2025-06-30

In [13]:
def make_chunks(d_start, d_end, days):
    chunks, current = [], d_start
    while current <= d_end:
        chunk_end = min(current + timedelta(days=days - 1), d_end)
        chunks.append((current, chunk_end))
        current = chunk_end + timedelta(days=1)
    return chunks

chunks = make_chunks(start, end, api["max_days_per_request"])
print(len(chunks), "requests")     
print(chunks[0], "...", chunks[-1]) 

79 requests
(datetime.date(2024, 1, 1), datetime.date(2024, 1, 7)) ... (datetime.date(2025, 6, 30), datetime.date(2025, 6, 30))


Setting up the API calls parameters to make sure that if data is not properly retreived, errors won't get stored as data. Also giving parameters a limit loading time
Then retreived data is called back as a dictionary and lists ready for python use

In [14]:
@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=2, min=2, max=60))
def fetch(url):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    return response.json()

Downloading the chunks with a printed warning in case it fails and a set up that allows to re run and skiped the already downloaded data

In [15]:
failed = []

for d_start, d_end in tqdm(chunks):
    out = RAW_DIR / f"{api['proceso']}_{d_start}_{d_end}.json"
    if out.exists():
        continue
    try:
        data = fetch(build_url(nodes, d_start, d_end))
        out.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    except Exception as e:
        failed.append((d_start, d_end, str(e)))
    time.sleep(1)

print("Done. Failed chunks:", len(failed))
failed

100%|██████████| 79/79 [00:00<00:00, 4770.85it/s]

Done. Failed chunks: 0


[]

Flatening all retreived data again and concatenating it and veryfying shape is in order and visualizing the first rows

In [16]:
raw_files = sorted(RAW_DIR.glob(f"{api['proceso']}_*.json"))
print(len(raw_files), "files")      

tables = []
for path in raw_files:
    data = json.loads(path.read_text(encoding="utf-8"))
    tables.append(parse_response(data))

df = pd.concat(tables, ignore_index=True)
print(df.shape)                   
df.head()

79 files
(118152, 8)


,clv_nodo,timestamp,fecha,hora,pml,pml_ene,pml_per,pml_cng
0,06AEO-115,2024-01-01 00:00:00,2024-01-01,1,279.92,370.71,-43.69,-47.1
1,06AEO-115,2024-01-01 01:00:00,2024-01-01,2,279.21,311.75,-32.54,0.0
2,06AEO-115,2024-01-01 02:00:00,2024-01-01,3,274.23,305.52,-31.28,0.0
3,06AEO-115,2024-01-01 03:00:00,2024-01-01,4,264.01,294.23,-30.22,0.0
4,06AEO-115,2024-01-01 04:00:00,2024-01-01,5,243.44,267.66,-24.22,0.0


Running an additional check

In [17]:
print("Nodes:", df["clv_nodo"].nunique())                    
print("From", df["timestamp"].min(), "to", df["timestamp"].max())
print("Rows per node:")
print(df["clv_nodo"].value_counts())                         
print("Duplicate rows:", df.duplicated(["clv_nodo", "timestamp"]).sum())  

Nodes: 9
From 2024-01-01 00:00:00 to 2025-06-30 23:00:00
Rows per node:
clv_nodo
06AEO-115    13128
06AHM-400    13128
06FRO-115    13128
06GRA-115    13128
06MON-115    13128
06MON-230    13128
06RGS-115    13128
06TKS-115    13128
06XOC-115    13128
Name: count, dtype: int64
Duplicate rows: 0


Number of nodes makes sense, we have the right count of values, and no duplicate values.
All good to save :)

In [18]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_parquet(PROCESSED_DIR / f"pml_{api['proceso'].lower()}.parquet", index=False)
print("Saved:", PROCESSED_DIR / f"pml_{api['proceso'].lower()}.parquet")

Saved: ..\data\processed\pml_mda.parquet
